In [1]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers
import scipy
from loguru import logger


In [2]:
verbose = False

delta = 2
m = 2

In [3]:
sampler = samplers.NoSignalingSampler(delta, m)

sampled_behavior = sampler.sample()

srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
    delta=delta,
    m=m,
    measured_behavior=sampled_behavior,
)

A_eq, b_eq = srns_set.get_equations(measured_behavior=sampled_behavior)

q_shape = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
).vector_shape[0]

lb = np.zeros(q_shape+1)
rb = np.ones(q_shape+1)
rb[0] = 2

bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]

c = np.zeros(q_shape+1)
c[0] = -1

if verbose:
    with np.printoptions(threshold=np.inf):
        print("A_eq")
        print(A_eq)
        print("b_eq")
        print(b_eq)
        print("bounds")
        print(bounds)
        print("c")
        print(c)
        print('\n\n---------\n\n')

print(f"Sampled behavior : {sampled_behavior}")
print(f"Sampled behavior is tested [{sampled_behavior.is_no_signaling()}] to being no signaling")


2025-05-19 17:22:34.550 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1, 32)
2025-05-19 17:22:34.552 | DEBUG    | no_signaling_sets:express_as_function_of_q:189 - Estimated memory complexity of lacking_betas: 56 bytes


Sampled behavior : Behavior:
Short path (z=S):
[[0.01354369 0.41057646 0.33426278 0.58895722]
 [0.4468133  0.04978053 0.43403974 0.17934529]
 [0.43524613 0.30977282 0.11452703 0.13139206]
 [0.10439688 0.22987019 0.11717045 0.10030542]]
Long path (z=L) :
[[0.22105067 0.33355379 0.33425371 0.50339265]
 [0.23930632 0.1268032  0.43404881 0.26490986]
 [0.22250436 0.32482355 0.10930133 0.15498469]
 [0.31713865 0.21481946 0.12239616 0.0767128 ]]
------------
Sampled behavior is tested [True] to being no signaling


In [4]:
import scipy.optimize

res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
    c=c,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
)

print(res.success)
print(-res.fun)
print(res.status)
print(res.x)

True
1.0572777508644178
0
[1.05727775 0.         0.41977392 0.33908916 0.60837193 0.45808633
 0.03831241 0.44458112 0.17529835 0.44585661 0.31319647 0.10676745
 0.12459846 0.09605706 0.2287172  0.10956227 0.09173126 0.21939252
 0.0733157  0.         0.26576387 0.11894705 0.44459071 0.11974676
 0.         0.00812508 0.10124242 0.2128044  0.         0.3209842
 0.0483     0.         0.06678729]


In [7]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(res.x[1:]), 0, 1),
)

test_tolerance = 1e-10

print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling(atol=test_tolerance)}] to being no signaling")


Yielded behavior is Behavior:
Short path (z=S):
[[0.         0.41977392 0.33908916 0.60837193]
 [0.45808633 0.03831241 0.44458112 0.17529835]
 [0.44585661 0.31319647 0.10676745 0.12459846]
 [0.09605706 0.2287172  0.10956227 0.09173126]]
Long path (z=L) :
[[0.21939252 0.0733157 ]
 [0.         0.26576387]
 [0.11894705 0.44459071]
 [0.11974676 0.        ]
 [0.00812508 0.10124242]
 [0.2128044  0.        ]
 [0.3209842  0.0483    ]
 [0.         0.06678729]]
------------
Yielded behavior is tested [True] to being no signaling


### Loop over samples to search for nontrivial alphas

In [6]:
# counter = 0
# logger.remove()

# while res.fun == 0:
#     sampled_behavior = sampler.sample()
#     srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
#         delta=delta,
#         m=m,
#         measured_behavior=sampled_behavior,
#     )
#     A_eq, b_eq = srns_set.get_equations(measured_behavior=sampled_behavior)
#     q_shape = behaviors.LatentSRNSBehavior(
#         delta=delta,
#         m=m,
#     ).vector_shape[0]
#     lb = np.zeros(q_shape+1)
#     rb = np.ones(q_shape+1)
#     rb[0] = 2
#     bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]
#     c = np.zeros(q_shape+1)
#     c[0] = 1

#     res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
#         c=c,
#         A_eq=A_eq,
#         b_eq=b_eq,
#         bounds=bounds,
#     )

#     counter += 1
#     if counter % 100 == 0:
#         print(f"Counter: {counter}")